# QNN 无监督学习教程：学习量子概率分布

本教程演示如何使用 `run_qnn_unsupervised` 训练参数化量子神经网络来学习给定样本背后的概率分布，并生成新样本：

1. **Autograd 路径**：Simulator + torch 自动微分，使用 **负对数似然 (NLL)** 损失
2. **Parameter-shift 路径**：Simulator 虚拟后端，使用 **MMD (Maximum Mean Discrepancy)** 损失
3. **对比分析**：两种路径的收敛行为与生成质量

> **注意**：autograd 路径直接计算 $P(b|\theta)$，损失为 NLL；parameter-shift 路径通过采样估计分布并用 MMD 衡量差异——两种方法优化不同的目标函数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from fieldqkit import QuantumHardwareClient
from fieldqkit.algorithms.qml import run_qnn_unsupervised
from fieldqkit.api.backend import Backend

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 构造训练数据

用 3 个量子比特构造一组 **有偏分布** 的样本：主要集中在 `|000⟩`、`|011⟩`、`|110⟩` 三种状态。

训练数据为 `(N, n_qubits)` 形状的整数数组，每行是一个 0/1 采样（big-endian：列 $i$ 对应 qubit $i$）。

In [ ]:
section("Data preparation")

num_qubits = 3
rng = np.random.default_rng(42)

# 目标分布：|000⟩ 40%, |011⟩ 35%, |110⟩ 25%
target_states = np.array([
    [0, 0, 0],
    [0, 1, 1],
    [1, 1, 0],
])
target_probs = np.array([0.40, 0.35, 0.25])

n_train = 800
n_test = 400

def sample_from_target(n, rng):
    indices = rng.choice(len(target_states), size=n, p=target_probs)
    return target_states[indices]

train_samples = sample_from_target(n_train, rng)
test_samples = sample_from_target(n_test, rng)

# 统计训练集频率
unique, counts = np.unique(train_samples, axis=0, return_counts=True)
print(f"Training samples: {n_train}")
print(f"Test samples:     {n_test}")
print(f"Num qubits:       {num_qubits}")
print(f"\nTraining distribution:")
for row, cnt in zip(unique, counts):
    bitstr = ''.join(str(b) for b in row)
    print(f"  |{bitstr}⟩ : {cnt}/{n_train} = {cnt/n_train:.1%}  (target: {target_probs[np.where((target_states == row).all(axis=1))[0][0]]:.0%})")

## 2) Autograd 路径（NLL 损失）

使用 torch 自动微分，直接计算量子态的 Born 概率 $P(b|\theta)$，最小化负对数似然：

$$\mathcal{L}_{\text{NLL}}(\theta) = -\frac{1}{N}\sum_{i=1}^{N} \log P(b_i|\theta)$$

这是精确梯度，无 shot noise，收敛快且稳定。

In [ ]:
section("Autograd (NLL loss)")

result_autograd = run_qnn_unsupervised(
    num_qubits=num_qubits,
    train_samples=train_samples,
    test_samples=test_samples,
    layers=3,
    max_iters=120,
    learning_rate=0.02,
    seed=42,
    gradient_method="autograd",
    gen_shots=2048,
)

print(f"\nBest loss (NLL): {result_autograd.best_loss:.6f}")
print(f"Parameters:      {len(result_autograd.best_params)}")
print(f"Generated:       {len(result_autograd.generated_samples)} samples")

## 3) Parameter-shift 路径（MMD 损失）

走后端流程：从电路中采样 shot，用 **Maximum Mean Discrepancy** (RBF 核) 衡量训练分布与生成分布的差异：

$$\text{MMD}^2(P, Q) = \mathbb{E}_{x,x'\sim P}[k(x,x')] - 2\mathbb{E}_{x\sim P, y\sim Q}[k(x,y)] + \mathbb{E}_{y,y'\sim Q}[k(y,y')]$$

其中 $k(x,y) = \exp(-\|x-y\|^2 / 2\sigma^2)$ 为 RBF 核。

parameter-shift 每步需执行 $1 + 2N_{\text{params}}$ 次电路（前向 + 正/负偏移），比 autograd 慢很多，因此减少迭代次数。

In [ ]:
section("Parameter-shift (MMD loss)")

client = QuantumHardwareClient()
backend = Backend("Simulator")

result_ps = run_qnn_unsupervised(
    num_qubits=num_qubits,
    train_samples=train_samples,
    test_samples=test_samples,
    layers=3,
    max_iters=40,
    learning_rate=0.005,
    seed=42,
    gradient_method="parameter-shift",
    client=client,
    backend=backend,
    chip_name="Simulator",
    shots=4096,
    mmd_sigma=1.0,
    gen_shots=2048,
)

print(f"\nBest loss (MMD): {result_ps.best_loss:.6f}")
print(f"Parameters:      {len(result_ps.best_params)}")
print(f"Generated:       {len(result_ps.generated_samples)} samples")

## 4) 生成分布分析

对比目标分布、训练集经验分布和两种方法生成的分布。

In [ ]:
def count_distribution(samples, num_qubits):
    """Return a dict mapping bitstring -> frequency."""
    samples = np.asarray(samples, dtype=int)
    all_labels = [''.join(str(b) for b in row) for row in samples]
    # all possible bitstrings
    all_possible = [format(i, f'0{num_qubits}b') for i in range(2**num_qubits)]
    freq = {bs: 0.0 for bs in all_possible}
    for label in all_labels:
        freq[label] += 1.0
    total = len(all_labels)
    return {k: v / total for k, v in freq.items()}

dist_train = count_distribution(train_samples, num_qubits)
dist_autograd = count_distribution(result_autograd.generated_samples, num_qubits)
dist_ps = count_distribution(result_ps.generated_samples, num_qubits)

# Target distribution
dist_target = {format(i, f'0{num_qubits}b'): 0.0 for i in range(2**num_qubits)}
for state, p in zip(target_states, target_probs):
    key = ''.join(str(b) for b in state)
    dist_target[key] = p

labels = sorted(dist_target.keys())
x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 1.5*width, [dist_target[l] for l in labels], width,
       label='Target', color='gray', alpha=0.7)
ax.bar(x - 0.5*width, [dist_train[l] for l in labels], width,
       label='Training set', color='steelblue', alpha=0.7)
ax.bar(x + 0.5*width, [dist_autograd[l] for l in labels], width,
       label='Autograd (NLL)', color='seagreen', alpha=0.7)
ax.bar(x + 1.5*width, [dist_ps[l] for l in labels], width,
       label='Param-shift (MMD)', color='crimson', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(['|' + l + '⟩' for l in labels], fontsize=9)
ax.set_ylabel('Probability')
ax.set_title('Distribution Comparison: Target vs Generated')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print numeric comparison
print(f"{'State':>8}  {'Target':>8}  {'Train':>8}  {'Autograd':>9}  {'ParamShift':>10}")
print("-" * 50)
for l in labels:
    t = dist_target[l]
    tr = dist_train[l]
    a = dist_autograd[l]
    p = dist_ps[l]
    if t > 0 or tr > 0.01 or a > 0.01 or p > 0.01:
        print(f"|{l}⟩  {t:8.3f}  {tr:8.3f}  {a:9.3f}  {p:10.3f}")

## 5) 收敛对比

**注意**：两种路径优化不同的损失函数（NLL vs MMD），数值不可直接对比，但可以观察各自的收敛趋势。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Autograd NLL ---
ax = axes[0]
ax.plot(result_autograd.loss_history, 'o-', ms=2, color='seagreen',
        label='train NLL')
if result_autograd.test_loss_history:
    ax.plot(result_autograd.test_loss_history, 's-', ms=2, color='darkorange',
            label='test NLL')
ax.set_xlabel('Iteration')
ax.set_ylabel('NLL Loss')
ax.set_title('Autograd: Negative Log-Likelihood')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- Parameter-shift MMD ---
ax = axes[1]
ax.plot(result_ps.loss_history, 'o-', ms=2, color='crimson',
        label='train MMD²')
if result_ps.test_loss_history:
    ax.plot(result_ps.test_loss_history, 's-', ms=2, color='darkorange',
            label='test MMD²')
ax.set_xlabel('Iteration')
ax.set_ylabel('MMD² Loss')
ax.set_title('Parameter-shift: MMD² (RBF kernel)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Autograd  — final NLL:  {result_autograd.loss_history[-1]:.6f}")
print(f"ParamShift — final MMD²: {result_ps.loss_history[-1]:.6f}")

## 6) 生成质量评估

计算生成分布与目标分布之间的 **总变差距离 (TVD)** 和 **KL 散度**：

$$\text{TVD}(P, Q) = \frac{1}{2}\sum_x |P(x) - Q(x)|$$

$$D_{\text{KL}}(P \| Q) = \sum_x P(x) \log\frac{P(x)}{Q(x)}$$

In [ ]:
def tvd(p_dict, q_dict):
    keys = set(p_dict) | set(q_dict)
    return 0.5 * sum(abs(p_dict.get(k, 0) - q_dict.get(k, 0)) for k in keys)

def kl_div(p_dict, q_dict, eps=1e-10):
    total = 0.0
    for k, p in p_dict.items():
        if p > 0:
            q = q_dict.get(k, 0) + eps
            total += p * np.log(p / q)
    return total

section("Generation quality")

print(f"{'Metric':>20}  {'Autograd':>10}  {'ParamShift':>11}")
print("-" * 46)
print(f"{'TVD vs target':>20}  {tvd(dist_target, dist_autograd):10.4f}  {tvd(dist_target, dist_ps):11.4f}")
print(f"{'KL(target||gen)':>20}  {kl_div(dist_target, dist_autograd):10.4f}  {kl_div(dist_target, dist_ps):11.4f}")

# Dominant fraction: fraction of generated samples in top-3 target states
target_labels = set(''.join(str(b) for b in row) for row in target_states)
auto_dominant = sum(dist_autograd.get(l, 0) for l in target_labels)
ps_dominant = sum(dist_ps.get(l, 0) for l in target_labels)
print(f"{'Dominant fraction':>20}  {auto_dominant:10.2%}  {ps_dominant:11.2%}")
print(f"{'(target)':>20}  {'100.00%':>10}  {'100.00%':>11}")

## 7) 实验开销分析

Parameter-shift 每次迭代的电路执行次数（无监督场景无多样本循环，比分类器更简洁）：

$$N_{\text{circuits/iter}} = 1 + 2 N_{\text{params}}$$

In [ ]:
section("Cost analysis")

n_params = len(result_ps.best_params)
n_iters_ps = len(result_ps.loss_history)
n_iters_ag = len(result_autograd.loss_history)

circuits_per_iter = 1 + 2 * n_params
total_circuits = circuits_per_iter * n_iters_ps

print(f"Num qubits:            {num_qubits}")
print(f"Parameters:            {n_params}")
print(f"\nAutograd path:")
print(f"  Iterations:          {n_iters_ag}")
print(f"  Circuit executions:  N/A (pure simulation)")
print(f"  Loss function:       NLL (exact P(b|θ))")
print(f"\nParameter-shift path:")
print(f"  Iterations:          {n_iters_ps}")
print(f"  Circuits/iter:       {circuits_per_iter}")
print(f"  Total circuits:      {total_circuits}")
print(f"  Transpilations:      1 (unified template)")
print(f"  Loss function:       MMD² (RBF kernel, σ={1.0})")

## 总结与下一步

| 方面 | Autograd (NLL) | Parameter-shift (MMD) |
|------|---------------|----------------------|
| **损失函数** | $-\frac{1}{N}\sum\log P(b_i|\theta)$ | $\text{MMD}^2_{\text{RBF}}(P_{\text{train}}, P_{\text{gen}})$ |
| **梯度** | torch 自动微分（精确） | 有限差分 + shot 估计（有噪声） |
| **适用场景** | 快速原型验证 | 真机部署 |
| **开销** | $O(2^n)$ 状态向量 | $O(N_{\text{params}})$ 电路/iter |

| 后续方向 | 说明 |
|---------|------|
| **增加比特数** | 测试 4-6 qubit 的可扩展性 |
| **调节 `mmd_sigma`** | RBF 带宽影响 MMD 的灵敏度 |
| **增加层数** | 更深的 ansatz 可表达更复杂的分布 |
| **真实硬件** | 替换 `Backend("Simulator")` 为真机后端 |
| **更复杂分布** | 如 GHZ 态、随机量子线路分布等 |